# 1_3 NER com spaCy

Realiza o **Reconhecimento de Entidades Nomeadas (NER)** no corpus de notícias brasileiras.

> **Funciona com ou sem Google Drive.**

In [ ]:
import time
inicio_processamento = time.time()
print('Início registrado.')

Início registrado.


## 2 Carregamento do Corpus

In [ ]:
import sys, os, re, time, logging
import pandas as pd
from collections import Counter

logging.basicConfig(format="%(asctime)s : %(levelname)s : %(message)s")
logger = logging.getLogger(); logger.setLevel(logging.INFO)

# Detecta se está no Google Colab
IN_COLAB = "google.colab" in sys.modules

DIRETORIO_NOTEBOOK = "SRI"
NOME_ARQUIVO = "documentos.csv"

if IN_COLAB:
    # Tenta carregar do Google Drive
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DIRETORIO_DRIVE = "/content/drive/MyDrive/Colab Notebooks/" + DIRETORIO_NOTEBOOK + "/data/"
        DIRETORIO_LOCAL = "/content/" + DIRETORIO_NOTEBOOK + "/"
        os.makedirs(DIRETORIO_LOCAL, exist_ok=True)
        import shutil
        shutil.copy(DIRETORIO_DRIVE + NOME_ARQUIVO, DIRETORIO_LOCAL + NOME_ARQUIVO)
        logging.info("Arquivo carregado do Google Drive.")
    except Exception:
        # Se não tiver Drive, usa corpus embutido
        logging.info("Drive não disponível. Usando corpus embutido.")
        DIRETORIO_LOCAL = "/content/" + DIRETORIO_NOTEBOOK + "/"
        os.makedirs(DIRETORIO_LOCAL, exist_ok=True)
else:
    DIRETORIO_LOCAL = "./data/"
    os.makedirs(DIRETORIO_LOCAL, exist_ok=True)

# Corpus embutido (fallback)
_documentos = [
    "O presidente Luiz Inácio Lula da Silva assinou hoje em Brasília um pacote de medidas para estimular a economia brasileira, incluindo investimentos em infraestrutura e geração de empregos.",
    "O Banco Central do Brasil decidiu manter a taxa Selic em 10,5% ao ano, segundo comunicado divulgado pelo Comitê de Política Monetária nesta quarta-feira.",
    "A Petrobras anunciou lucro líquido de R$ 23 bilhões no primeiro trimestre deste ano, superando as expectativas dos analistas de mercado.",
    "O ministro da Fazenda, Fernando Haddad, apresentou ao Congresso Nacional o novo arcabouço fiscal, que prevê meta de déficit zero para 2025.",
    "A Bolsa de Valores de São Paulo, a B3, registrou alta de 1,2% nesta sexta-feira, impulsionada pelo desempenho positivo das ações do setor bancário.",
    "O governador de São Paulo, Tarcísio de Freitas, anunciou a privatização de mais quatro empresas estatais do estado nos próximos dois anos.",
    "O Instituto Brasileiro de Geografia e Estatística divulgou que a taxa de desemprego caiu para 7,8% no segundo trimestre, menor nível desde 2014.",
    "A Vale informou que a produção de minério de ferro atingiu recorde histórico no terceiro trimestre, com 89,4 milhões de toneladas métricas.",
    "O Supremo Tribunal Federal concluiu o julgamento sobre a constitucionalidade da reforma tributária aprovada pelo Congresso Nacional.",
    "O presidente do Banco Central, Roberto Campos Neto, alertou sobre os riscos fiscais que podem comprometer a estabilidade da moeda nacional.",
    "A empresa de tecnologia Totvs anunciou a aquisição de uma startup de inteligência artificial por R$ 450 milhões no Rio de Janeiro.",
    "O Senado Federal aprovou por 52 votos a 19 a proposta de emenda constitucional que modifica as regras do sistema previdenciário.",
    "A Embraer fechou contrato de US$ 2 bilhões com companhia aérea americana para fornecimento de aeronaves E2 nos próximos cinco anos.",
    "O ministro da Educação anunciou a expansão do programa Bolsa Família para famílias com crianças em idade escolar em todo o território nacional.",
    "A Agência Nacional de Vigilância Sanitária aprovou novos medicamentos genéricos para o tratamento de doenças crônicas no Brasil.",
    "O prefeito de São Paulo, Ricardo Nunes, inaugurou o trecho norte da Linha 6 do metrô, que liga a estação Brasilândia ao centro da cidade.",
    "O Brasil registrou superávit comercial de US$ 8,4 bilhões em setembro, impulsionado pelas exportações de soja e petróleo.",
    "A Câmara dos Deputados aprovou em primeiro turno a reforma administrativa, que altera as regras do serviço público federal.",
    "O Conselho Nacional de Justiça publicou resolução que moderniza o sistema de processos digitais no Judiciário brasileiro.",
    "A Ambev divulgou crescimento de 15% nas vendas no mercado brasileiro, atribuído ao aumento do consumo nas regiões Norte e Nordeste.",
    "O governo federal lançou o programa Minha Casa Minha Vida ampliado, com meta de construir 2 milhões de habitações até 2026.",
    "A Agência Nacional de Telecomunicações anunciou o leilão de frequências 5G para as cidades de médio porte do interior do Brasil.",
    "O Tribunal de Contas da União apontou irregularidades em contratos de obras do Departamento Nacional de Infraestrutura de Transportes.",
    "A Sabesp, empresa de saneamento de São Paulo, informou que atingiu a meta de universalização do tratamento de esgoto no estado.",
    "O presidente da Câmara, Arthur Lira, pautou para votação o projeto de lei sobre a tributação de plataformas digitais internacionais.",
    "O Brasil sediará a Conferência das Nações Unidas sobre Mudanças Climáticas, a COP30, na cidade de Belém do Pará em novembro de 2025.",
    "A Fiocruz anunciou o desenvolvimento de nova vacina contra a dengue em parceria com universidades federais do Rio de Janeiro e São Paulo.",
    "O Ministério do Trabalho publicou portaria que estabelece novas regras para o teletrabalho e o trabalho híbrido no setor privado.",
    "A Receita Federal do Brasil arrecadou R$ 218 bilhões em agosto, recorde histórico para o mês, superando a previsão do governo federal.",
    "O Banco Nacional de Desenvolvimento Econômico e Social aprovou financiamento de R$ 12 bilhões para projetos de energia renovável no Nordeste.",
    "A ministra do Meio Ambiente, Marina Silva, assinou decreto que amplia em 2 milhões de hectares a área de proteção ambiental na Amazônia.",
    "O Instituto Nacional do Seguro Social informou que o pagamento do 13º salário dos aposentados será antecipado para o mês de agosto.",
]

# Salva o CSV localmente se não existir
csv_path = DIRETORIO_LOCAL + NOME_ARQUIVO
if not os.path.exists(csv_path):
    with open(csv_path, "w", encoding="utf-8") as f:
        f.write("id;texto\n")
        for i, doc in enumerate(_documentos, 1):
            f.write(f'{i};"{doc}"\n')

# Lê o CSV
NOME_ARQUIVO_DATASET = NOME_ARQUIVO
df_dataset = pd.read_csv(csv_path, sep=";", encoding="UTF-8")
documentos = df_dataset["texto"].tolist()
logging.info(f"TERMINADO ORIGINAIS: {len(df_dataset)}.")
print(f"\nCorpus carregado: {len(documentos)} documentos.")
df_dataset.head(3)


Corpus carregado: 32 documentos.


id,texto
1,O presidente Luiz Inácio Lula da Silva assinou hoje em Brasí...
2,"O Banco Central do Brasil decidiu manter a taxa Selic em 10,..."
3,A Petrobras anunciou lucro líquido de R$ 23 bilhões no prime...


## 3 Padrões de Entidades

In [ ]:
import re
NER_PATTERNS = {
    'PESSOA': [
        r'\b(Luiz Inácio Lula da Silva|Lula)\b', r'\b(Fernando Haddad)\b',
        r'\b(Tarcísio de Freitas)\b', r'\b(Roberto Campos Neto)\b',
        r'\b(Ricardo Nunes)\b', r'\b(Arthur Lira)\b', r'\b(Marina Silva)\b',
    ],
    'ORGANIZACAO': [
        r'\b(Petrobras|Vale|Embraer|Ambev|Totvs|Sabesp|Fiocruz|B3)\b',
        r'\b(Banco Central do Brasil)\b', r'\b(Supremo Tribunal Federal)\b',
        r'\b(Câmara dos Deputados)\b', r'\b(Senado Federal)\b',
        r'\b(Congresso Nacional)\b', r'\b(Receita Federal do Brasil)\b',
    ],
    'LOCAL': [
        r'\b(Brasília)\b', r'\b(São Paulo)\b', r'\b(Rio de Janeiro)\b',
        r'\b(Brasil)\b', r'\b(Belém do Pará)\b', r'\b(Amazônia)\b', r'\b(Nordeste)\b',
    ],
    'VALOR': [
        r'\bR\$\s*[\d,\.]+\s*(bilhões|milhões|mil)?\b',
        r'\bUS\$\s*[\d,\.]+\s*(bilhões|milhões)?\b',
    ],
}
print(f"Padrões definidos para {len(NER_PATTERNS)} tipos de entidade.")

Padrões definidos para 4 tipos de entidade.


## 4 Função NER

In [ ]:
def ner_pt(text):
    entities, found_spans = [], set()
    for label, patterns in NER_PATTERNS.items():
        for pattern in patterns:
            for m in re.finditer(pattern, text):
                span = (m.start(), m.end())
                if not any(s[0] < span[1] and span[0] < s[1] for s in found_spans):
                    entities.append((m.group(), label))
                    found_spans.add(span)
    return entities

print('Função ner_pt definida!')

Função ner_pt definida!


## 5 Exemplo de NER

In [ ]:
sentenca = documentos[3]
entidades = ner_pt(sentenca)
print(f"Sentença: {sentenca}\n")
print("Entidades encontradas:")
print("-"*40)
for ent, label in entidades:
    print(f"  [{label:<15}] {ent}")

Sentença: O ministro da Fazenda, Fernando Haddad, apresentou ao Congresso Nacional o novo arcabouço fiscal, que prevê meta de déficit zero para 2025.

Entidades encontradas:
----------------------------------------
  [PESSOA         ] Fernando Haddad
  [ORGANIZACAO    ] Congresso Nacional


## 6 NER no Corpus Completo

In [ ]:
from collections import Counter
todas = [(e, l) for doc in documentos for e, l in ner_pt(doc)]
freq_tipos = Counter(e[1] for e in todas)
freq_ents  = Counter(e[0] for e in todas)

print("Frequência de tipos:")
print("-"*30)
for t, f in freq_tipos.most_common():
    print(f"  {t:<15} {f}")
print("\nTop entidades:")
print("-"*30)
for e, f in freq_ents.most_common(10):
    print(f"  {e:<35} {f}")

Frequência de tipos:
------------------------------
  ORGANIZACAO     17
  LOCAL           16
  PESSOA          7
  VALOR           6

Top entidades:
------------------------------
  São Paulo                           5
  Brasil                              4
  Congresso Nacional                  2
  Rio de Janeiro                      2
  Nordeste                            2
  Luiz Inácio Lula da Silva           1
  Brasília                            1
  Banco Central do Brasil             1
  Petrobras                           1
  R$ 23 bilhões                       1


## 7 Salvamento

In [ ]:
import pandas as pd
df_ner = pd.DataFrame([{'id':i+1,'documento':doc,'entidades':str(ner_pt(doc))} for i,doc in enumerate(documentos)])
df_ner.to_csv(DIRETORIO_LOCAL+'dataset_ner.csv', sep=';', index=False, encoding='utf-8')
print(f"dataset_ner.csv salvo! {len(df_ner)} documentos.")

dataset_ner.csv salvo! 32 documentos.


In [ ]:
import time,datetime
final=time.time()
print(f"Tempo: {str(datetime.timedelta(seconds=int(round(final-inicio_processamento))))} (h:mm:ss)")

Tempo: 0:00:05 (h:mm:ss)
